In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/vision_lab"
SEQUENCE = "structure_texture"   # must match what you used in 01_prep

synced_dir = f"{BASE}/{SEQUENCE}/synced_data"
raw_dir = f"{synced_dir}/rgb_raw"
depth_gt_dir = f"{synced_dir}/depth"

import os
print(f"RGB frames found: {len(os.listdir(raw_dir))}")
print(f"Depth frames found: {len(os.listdir(depth_gt_dir))}")

Mounted at /content/drive
RGB frames found: 1057
Depth frames found: 1057


In [ ]:
import torch
import cv2
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

model = torch.hub.load("intel-isl/MiDaS", "DPT_Large", trust_repo=True)
transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).dpt_transform
model.to(device).eval()
print("DPT_Large loaded and ready")

Running on cuda
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Downloading: "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_large_384.pt" to /root/.cache/torch/hub/checkpoints/dpt_large_384.pt


100%|██████████| 1.28G/1.28G [00:08<00:00, 155MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


DPT_Large loaded and ready


In [ ]:
def raw_passthrough(img_bgr):
    return img_bgr

def standard_clahe(img_bgr, clip=3.0, tile=8):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    l_enh = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

def ssr_retinex(img_bgr, sigma=80):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_f = np.log1p(l.astype(np.float32))
    illumination = cv2.GaussianBlur(l_f, (0, 0), sigma)
    retinex = l_f - illumination
    l_enh = np.clip(np.expm1(retinex + illumination * 1.5), 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

def lime_enhance(img_bgr, gamma=0.6):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_f = l.astype(np.float32) / 255.0
    l_enh = (np.power(l_f, gamma) * 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

def base_only(img_bgr, shading_gain=2.5, noise_suppression=0.3):
    # your BASE function, lines 619-628, unchanged
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_f = l.astype(np.float32)
    illumination = cv2.bilateralFilter(l_f, d=15, sigmaColor=75, sigmaSpace=75)
    reflectance = l_f - illumination
    l_enh = (illumination * shading_gain) + (reflectance * noise_suppression)
    l_enh = np.clip(l_enh, 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

def reflectance_clahe(img_bgr, clip=3.0, tile=8):
    # ABLATION: CLAHE on the WRONG component (reflectance, not illumination)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_f = l.astype(np.float32)
    illumination = cv2.bilateralFilter(l_f, d=15, sigmaColor=75, sigmaSpace=75)
    reflectance = l_f - illumination
    refl_uint8 = np.clip(reflectance + 128, 0, 255).astype(np.uint8)  # shift to valid range
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    refl_enh = clahe.apply(refl_uint8).astype(np.float32) - 128
    l_enh = np.clip(illumination + refl_enh, 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

def base_clahe_hybrid(img_bgr, shading_gain=2.0, noise_suppression=0.2):
    # your proposed method, lines 630-644, unchanged
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_f = l.astype(np.float32)
    illumination = cv2.bilateralFilter(l_f, d=15, sigmaColor=75, sigmaSpace=75)
    reflectance = l_f - illumination
    illum_uint8 = np.clip(illumination, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced_illum = clahe.apply(illum_uint8).astype(np.float32)
    l_enh = (enhanced_illum * shading_gain) + (reflectance * noise_suppression)
    l_enh = np.clip(l_enh, 0, 255).astype(np.uint8)
    return cv2.cvtColor(cv2.merge([l_enh, a, b]), cv2.COLOR_LAB2BGR)

METHODS = {
    "raw": raw_passthrough,
    "ssr_retinex": ssr_retinex,
    "lime": lime_enhance,
    "clahe_std": standard_clahe,
    "base_only": base_only,
    "reflectance_clahe": reflectance_clahe,
    "hybrid": base_clahe_hybrid,
}
print(f"{len(METHODS)} methods ready: {list(METHODS.keys())}")

7 methods ready: ['raw', 'ssr_retinex', 'lime', 'clahe_std', 'base_only', 'reflectance_clahe', 'hybrid']


In [ ]:
from tqdm import tqdm

def run_dpt(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        prediction = model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1), size=img_bgr.shape[:2],
            mode="bicubic", align_corners=False,
        ).squeeze()
    disp = prediction.cpu().numpy()
    return 1.0 / (disp + 1e-8)

all_frames = sorted([f for f in os.listdir(raw_dir) if f.endswith('.png')])
sampled = all_frames[::10]
print(f"Processing {len(sampled)} of {len(all_frames)} frames")

out_dirs = {name: f"vision_workspace/depths/{name}" for name in METHODS}
for d in out_dirs.values():
    os.makedirs(d, exist_ok=True)

for fname in tqdm(sampled, desc="Running 7-way inference"):
    img = cv2.imread(os.path.join(raw_dir, fname))
    if img is None:
        continue
    save_name = fname.replace(".png", ".npy")
    for name, fn in METHODS.items():
        processed = fn(img)
        depth = run_dpt(processed)
        np.save(os.path.join(out_dirs[name], save_name), depth)

torch.cuda.empty_cache()
print("Done")

Processing 106 of 1057 frames


Running 7-way inference: 100%|██████████| 106/106 [05:03<00:00,  2.86s/it]

Done


In [ ]:
import shutil
dest = f"{BASE}/{SEQUENCE}/depths"
shutil.copytree("vision_workspace/depths", dest, dirs_exist_ok=True)
print(f"Saved to {dest}")

Saved to /content/drive/MyDrive/vision_lab/structure_texture/depths
